In [3]:
# For loading HF_Token
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [ ]:
# Import stuff
import torch
import torch.nn as nn
import einops
from transformer_lens.hook_points import HookPoint

import tqdm.auto as tqdm
import plotly.express as px
from transformer_lens.model_bridge import TransformerBridge

from jaxtyping import Float
from functools import partial

/home/fin/doccuments/work/mech-interp-mini-projects/ioi-activation-patching/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Setting plotly renderer to work for notebooks
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [6]:
import circuitsvis as cv

# Testing that the library works
cv.examples.hello("Fin")

In [7]:
# Turn off auto differentation to save memory
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [8]:
from transformer_lens import utils


def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(y=utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(renderer)


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs).show(renderer)

/tmp/ipykernel_578696/2314955687.py:1: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import utils


In [9]:
device = utils.get_device()

In [15]:
model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 25334.95it/s]


In [48]:
def logits_to_logit_diff(logits: torch.Tensor, clean_answer = " Paris", corrupted_answer = " Madrid") -> float:
    correct_answer_token = model.to_single_token(clean_answer)
    corrupted_answer_token = model.to_single_token(corrupted_answer)
    return float(logits[0, -1, correct_answer_token] - logits[0, -1, corrupted_answer_token])
    

In [59]:
clean_input = "The capital of France is"
clean_tokens = model.to_tokens(clean_input)
clean_logits, clean_cache = model.run_with_cache(clean_input)
clean_logit_diff = logits_to_logit_diff(clean_logits)


corrupted_input = "The capital of Spain is"
corrupted_tokens = model.to_tokens(corrupted_input)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_input)
corrupted_logit_diff= logits_to_logit_diff(corrupted_logits)

In [ ]:
def patching_hook(value: torch.Tensor, hook: HookPoint, head_index: int) -> torch.Tensor:
    clean_head = clean_cache[hook.name]
    value[:, :, head_index, :] = clean_head[:, :, head_index, :]
    return value

num_layers = model.cfg.n_layers
heads_per_layer = model.cfg.n_heads

results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in tqdm.tqdm(range(num_layers)):
    for head in range(heads_per_layer):
        hook_fn = partial(patching_hook, head_index=head)
        patched_logits = model.run_with_hooks(
            corrupted_input,
            fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)]
        )
        
        patched_logits_diff = logits_to_logit_diff(patched_logits)
        results[layer, head] = (patched_logits_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

100%|██████████| 12/12 [00:01<00:00, 11.94it/s]


In [ ]:
imshow(
    results,
    xaxis="Head",
    yaxis="Layer",
    title="Denoising: Loss by Layer and Head",
)

In [80]:
def patching_hook(value: torch.Tensor, hook: HookPoint, head_index: int) -> torch.Tensor:
    corrupted_head = corrupted_cache[hook.name]
    value[:, :, head_index, :] = corrupted_head[:, :, head_index, :]
    return value

num_layers = model.cfg.n_layers
heads_per_layer = model.cfg.n_heads

results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in tqdm.tqdm(range(num_layers)):
    for head in range(heads_per_layer):
        hook_fn = partial(patching_hook, head_index=head)
        patched_logits = model.run_with_hooks(
            clean_input,
            fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)]
        )
        
        patched_logits_diff = logits_to_logit_diff(patched_logits)
        results[layer, head] = (clean_logit_diff - patched_logits_diff) / (clean_logit_diff - corrupted_logit_diff)

100%|██████████| 12/12 [00:00<00:00, 12.22it/s]


In [81]:
imshow(
    results,
    xaxis="Head",
    yaxis="Layer",
    title="Noising: Loss by Layer and Head",
)